# Reconstruct: 1 イベント 1 行の再構成ツリー

TPC の反跳電子のエネルギー $K_e$(`TPC_Calibration`)と、シンチレータの散乱ガンマ線のエネルギー $E_g$・吸収点 $P_g$(`PSA_Event`)から、
コンプトン散乱の運動学の量を求めて、`Reconstruct/single_hit` に 1 イベント 1 行で書きます。

| 書くイベント | シンチレータの 1 ヒット(`PSA_SingleHit`、SMILE-2+ なら `HA2p_SingleHit`)と TPC のエネルギー(`TPC_Ee_Valid`)の両方があるもの。運動学が合わない($|\cos\alpha_{kin}| > 1$)ものは `NotConformEnergy` で除く |
|---|---|

| 列 | 式・中身 |
|---|---|
| `evt_id` | トリガーカウンタ(`ReadData`) |
| `Eg`, `Ke`, `E0` | $E_0 = E_g + K_e$ [keV] |
| `Pg[3]` | 吸収点 [mm] |
| `cos_phi` | 散乱角: $\cos\phi = 1 - m_ec^2 (1/E_g - 1/E_0)$ |
| `cos_psi_kin` | 反跳角: $\cos\psi = (1 + m_ec^2/E_0)\sqrt{K_e / (K_e + 2 m_ec^2)}$ |
| `cos_alpha_kin` | 散乱ガンマ線と反跳電子のなす角: $\cos\alpha = (1 - m_ec^2/E_g)\sqrt{K_e / (K_e + 2 m_ec^2)}$ |
| `max_cos_alpha` | $k = E_0/m_ec^2$ として $\frac{k-1}{k+2}\sqrt{\frac{k^2-1}{k(k+2)}}$ |

> **今の状態**: 電子の方向 `Ve`(`TOT_Skewness`)を使った幾何の $\alpha$ や到来方向はまだ計算していません(EricaNext_try の Reconstruct にはある)。
> TPC のゲインマップが無いので $K_e$ は ADU × サンプル、シンチレータのジオメトリ `geometry.csv` も無いので、今のままでは `single_hit` は空です。
> 下では、[10_PSA_Event](../10_PSA_Event/PSA_Event.ipynb) と同じ**例の値**(ジオメトリとシンチのエネルギー)で、仕組みだけを見ます。

In [1]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有、git には入れない)
CACHE.mkdir(exist_ok=True)

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    dt = np.float32 if h.InheritsFrom("TArrayF") else np.float64   # TH2F / TH2D
    a = np.frombuffer(h.GetArray(), dtype=dt, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2).astype(float)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def th1(h):
    """ROOT の TH1 -> (中身, ビンの中心)"""
    n = h.GetNbinsX()
    return np.array([h.GetBinContent(i) for i in range(1, n + 1)]), np.array([h.GetBinCenter(i) for i in range(1, n + 1)])

def vec(column):
    """RDataFrame.AsNumpy() の配列型の列 -> numpy の配列のリスト"""
    return [np.asarray(v) for v in column]

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データとチェーン

TPC とシンチレータを一緒に読むので、ETCC のラン `20261102_run04`(NAS、211 イベント)を、[10_PSA_Event](../10_PSA_Event/PSA_Event.ipynb) と同じく
両方の storage のファイルへのリンクを集めたディレクトリで流します。設定ファイルは `smile3_reconstruct` です。

In [2]:
DATA = Path(os.environ.get("SMILE3_FM_DATA", "/mnt/nas_eureka/SMILE-3FM_data"))
RUN = "20261102_run04"
RUN_DIR = CACHE / f"etcc_{RUN}"
RUN_DIR.mkdir(exist_ok=True)
for d in [DATA / "20261102" / "calltt" / RUN, DATA / "20261102" / "killua" / RUN]:
    for p in list(d.glob("ha*_*.dat")) + list(d.glob("tpc_enc_*.raw")):
        if not (RUN_DIR / p.name).exists():
            (RUN_DIR / p.name).symlink_to(p)
EX = CACHE / "example_psa_cal"   # 10_PSA_Event で作った例の値(なければ作る)
EX.mkdir(exist_ok=True)
boards = sorted({int(p.name[2:4]) for p in RUN_DIR.glob("ha*_*.dat")})
with open(EX / "geometry.csv", "w") as g:
    g.write("# EXAMPLE values for the tutorial, not the real geometry\nboard,ch,x0,y0,z0,ux,uy,uz,vx,vy,vz,pitch\n")
    for b in boards:
        for c in range(6):
            g.write(f"{b},{c},{100 * b},{50 * c},200,1,0,0,0,1,0,6\n")
for b in boards:
    with open(EX / f"energy_ha{b:02d}.csv", "w") as e:
        e.write("ch,n,m,p0,p1\n" + "".join(f"{c},{n},{m},0,0.01\n" for c in range(6) for n in range(1, 9) for m in range(1, 9)))
OUT = CACHE / f"reconstruct_psa_{RUN}_example.root"
cmd = [sys.executable, str(RUN_CHAIN), "smile3_reconstruct", "-o", str(OUT), "--data-dir", str(RUN_DIR),
       "--set", f"PSA_Calibration.energy_dir={EX}", "--set", f"PSA_Event.geometry_file={EX / 'geometry.csv'}"]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True); r.check_returncode()
for l in r.stdout.splitlines():
    if l.split()[:1] and l.split()[0] in ("PSA_SingleHit", "PSA_MultiHit", "PSA_NoHits", "TPC_Ee_Valid", "NotConformEnergy"):
        print(l)

/usr/bin/python3 /home/onoda/ANLNext/Erica_git_dev/EricaNext/run_script/run_chain.py smile3_reconstruct -o /home/onoda/ANLNext/Erica_git_dev/EricaNext/tutorials/cache/reconstruct_psa_20261102_run04_example.root --data-dir /home/onoda/ANLNext/Erica_git_dev/EricaNext/tutorials/cache/etcc_20261102_run04 --set PSA_Calibration.energy_dir=/home/onoda/ANLNext/Erica_git_dev/EricaNext/tutorials/cache/example_psa_cal --set PSA_Event.geometry_file=/home/onoda/ANLNext/Erica_git_dev/EricaNext/tutorials/cache/example_psa_cal/geometry.csv


NotConformEnergy                                           26                0
PSA_MultiHit                                              139              139
PSA_NoHits                                                 10               10
PSA_SingleHit                                              62               36
TPC_Ee_Valid                                              202              176


## 2. single_hit

`PSA_SingleHit` かつ `TPC_Ee_Valid` のイベントのうち、運動学の合うものが書かれます。$K_e$ はゲインマップが無いので ADU × サンプル、$E_g$ は例の値なので、数値そのものに意味はありません。

In [3]:
s = ROOT.RDataFrame("Reconstruct/single_hit", str(OUT)).AsNumpy(["evt_id", "Eg", "Ke", "E0", "Pg", "cos_phi", "cos_psi_kin", "cos_alpha_kin", "max_cos_alpha"])
print(len(s["evt_id"]), "rows")
me = 510.99895
for i in range(min(5, len(s["evt_id"]))):
    eg, ke = s["Eg"][i], s["Ke"][i]; e0 = eg + ke
    print(f"evt_id {s['evt_id'][i]:4d}: Eg {eg:7.1f} Ke {ke:8.1f} E0 {e0:8.1f}  cos_phi {s['cos_phi'][i]:+.3f} (= {1 - me * (1 / eg - 1 / e0):+.3f})"
          f"  cos_alpha_kin {s['cos_alpha_kin'][i]:+.3f} (= {(1 - me / eg) * np.sqrt(ke / (ke + 2 * me)):+.3f})")

33 rows
evt_id    3: Eg   710.2 Ke   1183.7 E0   1893.9  cos_phi +0.550 (= +0.550)  cos_alpha_kin +0.205 (= +0.205)
evt_id    4: Eg   625.0 Ke   3330.9 E0   3955.8  cos_phi +0.312 (= +0.312)  cos_alpha_kin +0.160 (= +0.160)
evt_id    5: Eg   343.1 Ke   5672.6 E0   6015.7  cos_phi -0.404 (= -0.404)  cos_alpha_kin -0.450 (= -0.450)
evt_id    7: Eg   150.4 Ke     41.6 E0    192.0  cos_phi +0.264 (= +0.264)  cos_alpha_kin -0.474 (= -0.474)
evt_id    8: Eg   304.2 Ke   4414.6 E0   4718.8  cos_phi -0.571 (= -0.571)  cos_alpha_kin -0.612 (= -0.612)


## 3. パラメータ

`Reconstruct` にパラメータはありません。$m_ec^2 = 510.99895$ keV です。